In [1]:
# ============================================================
# TASK 5 - DATA QUALITY, RECONCILIATION & RELIABILITY ANALYSIS
# Python Analysis - Final Version
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. FILE PATH
# ------------------------------------------------------------

file_path = "/content/Cleaned_Dataset.xlsx"

print("=" * 70)
print("TASK 5 - PYTHON DATA ANALYSIS")
print("=" * 70)

# Check file
if not os.path.exists(file_path):
    print("❌ FILE NOT FOUND")
    print("Files available in Colab:")
    print(os.listdir("/content"))
    raise FileNotFoundError(
        "Please upload Cleaned_Dataset.xlsx to Colab."
    )

print("✅ File found:", file_path)

# ------------------------------------------------------------
# 2. READ EXCEL
# ------------------------------------------------------------

excel = pd.ExcelFile(file_path)

print("\nAvailable Sheets:")
for i, sheet in enumerate(excel.sheet_names, 1):
    print(f"{i}. {sheet}")

# ------------------------------------------------------------
# 3. DETECT SOURCE & WAREHOUSE SHEETS
# ------------------------------------------------------------

source_sheet = None
warehouse_sheet = None

for sheet in excel.sheet_names:

    name = sheet.lower().replace(" ", "_")

    if "warehouse" in name:
        warehouse_sheet = sheet

    elif "source" in name:
        source_sheet = sheet

# Fallback
if source_sheet is None:
    source_sheet = excel.sheet_names[0]

if warehouse_sheet is None and len(excel.sheet_names) > 1:
    warehouse_sheet = excel.sheet_names[1]

print("\nSource Sheet    :", source_sheet)
print("Warehouse Sheet :", warehouse_sheet)

# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

source_df = pd.read_excel(
    file_path,
    sheet_name=source_sheet
)

warehouse_df = pd.read_excel(
    file_path,
    sheet_name=warehouse_sheet
)

# Clean column names
source_df.columns = (
    source_df.columns
    .astype(str)
    .str.strip()
)

warehouse_df.columns = (
    warehouse_df.columns
    .astype(str)
    .str.strip()
)

print("\nSource Shape    :", source_df.shape)
print("Warehouse Shape :", warehouse_df.shape)

print("\nSource Columns:")
print(list(source_df.columns))

print("\nWarehouse Columns:")
print(list(warehouse_df.columns))

# ------------------------------------------------------------
# 5. STANDARDIZE DATA
# ------------------------------------------------------------

for df in [source_df, warehouse_df]:

    # Text columns
    for col in ["Record_ID", "City", "Source_System", "Status"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
            )

    # Amount
    if "Amount" in df.columns:
        df["Amount"] = pd.to_numeric(
            df["Amount"],
            errors="coerce"
        )

    # Dates
    if "Event_Date" in df.columns:
        df["Event_Date"] = pd.to_datetime(
            df["Event_Date"],
            errors="coerce"
        )

    if "Last_Updated" in df.columns:
        df["Last_Updated"] = pd.to_datetime(
            df["Last_Updated"],
            errors="coerce"
        )

# Standardize Status
for df in [source_df, warehouse_df]:

    if "Status" in df.columns:
        df["Status"] = (
            df["Status"]
            .str.strip()
            .str.title()
        )

print("\n✅ Data standardization completed.")

# ------------------------------------------------------------
# 6. BASIC DATA QUALITY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. DATA QUALITY CHECK")
print("=" * 70)

print("Source Records     :", len(source_df))
print("Warehouse Records  :", len(warehouse_df))

source_duplicates = source_df.duplicated().sum()
warehouse_duplicates = warehouse_df.duplicated().sum()

print("Source Duplicates  :", source_duplicates)
print("Warehouse Duplicates:", warehouse_duplicates)

# ------------------------------------------------------------
# 7. MISSING VALUES
# ------------------------------------------------------------

source_missing = pd.DataFrame({
    "Column": source_df.columns,
    "Missing_Values": [
        source_df[col].isna().sum()
        for col in source_df.columns
    ]
})

source_missing["Missing_%"] = (
    source_missing["Missing_Values"]
    / len(source_df)
    * 100
).round(2)

print("\nMissing Value Report:")
display(source_missing)

# ------------------------------------------------------------
# 8. COMPLETENESS
# ------------------------------------------------------------

total_cells = (
    source_df.shape[0]
    * source_df.shape[1]
)

missing_cells = source_df.isna().sum().sum()

overall_completeness = (
    (total_cells - missing_cells)
    / total_cells
    * 100
)

print(
    "Overall Data Completeness:",
    round(overall_completeness, 2),
    "%"
)

# ------------------------------------------------------------
# 9. SOURCE VS WAREHOUSE RECORD RECONCILIATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. SOURCE VS WAREHOUSE RECONCILIATION")
print("=" * 70)

source_ids = set(
    source_df["Record_ID"].dropna()
)

warehouse_ids = set(
    warehouse_df["Record_ID"].dropna()
)

matched_ids = source_ids.intersection(
    warehouse_ids
)

missing_in_warehouse = (
    source_ids - warehouse_ids
)

warehouse_only = (
    warehouse_ids - source_ids
)

source_count = len(source_ids)
warehouse_count = len(warehouse_ids)
matched_count = len(matched_ids)

reconciliation_rate = (
    matched_count
    / source_count
    * 100
    if source_count > 0 else 0
)

print("Source Records          :", source_count)
print("Warehouse Records       :", warehouse_count)
print("Matched Records         :", matched_count)
print("Missing in Warehouse    :", len(missing_in_warehouse))
print("Warehouse Only Records  :", len(warehouse_only))
print(
    "Reconciliation Rate     :",
    round(reconciliation_rate, 2),
    "%"
)

# ------------------------------------------------------------
# 10. RECORD ID RECONCILIATION DETAIL
# ------------------------------------------------------------

reconciliation_detail = pd.DataFrame({
    "Record_ID": list(
        source_ids.union(warehouse_ids)
    )
})

reconciliation_detail["In_Source"] = (
    reconciliation_detail["Record_ID"]
    .isin(source_ids)
)

reconciliation_detail["In_Warehouse"] = (
    reconciliation_detail["Record_ID"]
    .isin(warehouse_ids)
)

reconciliation_detail["Reconciliation_Status"] = np.select(
    [
        (
            reconciliation_detail["In_Source"]
            & reconciliation_detail["In_Warehouse"]
        ),
        (
            reconciliation_detail["In_Source"]
            & ~reconciliation_detail["In_Warehouse"]
        ),
        (
            ~reconciliation_detail["In_Source"]
            & reconciliation_detail["In_Warehouse"]
        )
    ],
    [
        "Matched",
        "Missing in Warehouse",
        "Warehouse Only"
    ],
    default="Unknown"
)

# ------------------------------------------------------------
# 11. COMMON COLUMN COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. DETAIL FIELD RECONCILIATION")
print("=" * 70)

# Only compare columns which genuinely exist in BOTH datasets
comparison_columns = [
    col
    for col in [
        "Event_Date",
        "City",
        "Source_System",
        "Amount",
        "Status"
    ]
    if col in source_df.columns
    and col in warehouse_df.columns
]

print("Common fields being compared:")
print(comparison_columns)

comparison = source_df.merge(
    warehouse_df,
    on="Record_ID",
    how="inner",
    suffixes=("_Source", "_Warehouse")
)

# ------------------------------------------------------------
# 12. FIELD MATCH FLAGS
# ------------------------------------------------------------

match_columns = []

for col in comparison_columns:

    source_col = f"{col}_Source"
    warehouse_col = f"{col}_Warehouse"
    match_col = f"{col}_Match"

    if col == "Amount":

        comparison[match_col] = np.isclose(
            pd.to_numeric(
                comparison[source_col],
                errors="coerce"
            ),
            pd.to_numeric(
                comparison[warehouse_col],
                errors="coerce"
            ),
            equal_nan=True
        )

    elif col in ["Event_Date", "Last_Updated"]:

        comparison[match_col] = (
            pd.to_datetime(
                comparison[source_col],
                errors="coerce"
            )
            ==
            pd.to_datetime(
                comparison[warehouse_col],
                errors="coerce"
            )
        )

    else:

        comparison[match_col] = (
            comparison[source_col]
            .astype("string")
            .str.strip()
            ==
            comparison[warehouse_col]
            .astype("string")
            .str.strip()
        )

    match_columns.append(match_col)

# Overall field match
if match_columns:

    comparison["Overall_Field_Match"] = (
        comparison[match_columns]
        .all(axis=1)
    )

else:

    comparison["Overall_Field_Match"] = True

mismatch_df = comparison[
    comparison["Overall_Field_Match"] == False
].copy()

print(
    "\nMatched Detail Records:",
    comparison["Overall_Field_Match"].sum()
)

print(
    "Mismatched Detail Records:",
    len(mismatch_df)
)

# ------------------------------------------------------------
# 13. DATA RISK CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. DATA RISK CHECK")
print("=" * 70)

negative_amounts = 0
missing_amounts = 0
missing_dates = 0
missing_last_updated = 0

if "Amount" in source_df.columns:
    negative_amounts = (
        source_df["Amount"] < 0
    ).sum()

    missing_amounts = (
        source_df["Amount"].isna()
    ).sum()

if "Event_Date" in source_df.columns:
    missing_dates = (
        source_df["Event_Date"].isna()
    ).sum()

if "Last_Updated" in source_df.columns:
    missing_last_updated = (
        source_df["Last_Updated"].isna()
    ).sum()

print("Negative Amounts       :", negative_amounts)
print("Missing Amounts        :", missing_amounts)
print("Missing Event Dates    :", missing_dates)
print("Missing Last Updated   :", missing_last_updated)

# ------------------------------------------------------------
# 14. FRESHNESS CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. DATA FRESHNESS")
print("=" * 70)

if "Last_Updated" in source_df.columns:

    latest_update = source_df["Last_Updated"].max()

    print("Latest Last_Updated:", latest_update)

    if pd.isna(latest_update):

        freshness_status = "Unknown"

    else:

        # Use latest available record date
        current_reference = source_df["Last_Updated"].max()

        freshness_status = "Available"

else:

    latest_update = None
    freshness_status = "Not Available"

print("Freshness Status:", freshness_status)

# ------------------------------------------------------------
# 15. BUSINESS SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. BUSINESS SUMMARY")
print("=" * 70)

total_amount = (
    source_df["Amount"].sum()
    if "Amount" in source_df.columns
    else 0
)

average_amount = (
    source_df["Amount"].mean()
    if "Amount" in source_df.columns
    else 0
)

if "Status" in source_df.columns:

    status_summary = (
        source_df["Status"]
        .value_counts()
        .reset_index()
    )

    status_summary.columns = [
        "Status",
        "Record_Count"
    ]

else:

    status_summary = pd.DataFrame()

if "City" in source_df.columns:

    city_summary = (
        source_df.groupby("City", dropna=False)
        .agg(
            Record_Count=("Record_ID", "count"),
            Total_Amount=("Amount", "sum"),
            Average_Amount=("Amount", "mean")
        )
        .reset_index()
        .sort_values(
            "Total_Amount",
            ascending=False
        )
    )

else:

    city_summary = pd.DataFrame()

if "Source_System" in source_df.columns:

    source_system_summary = (
        source_df.groupby(
            "Source_System",
            dropna=False
        )
        .agg(
            Record_Count=("Record_ID", "count"),
            Total_Amount=("Amount", "sum"),
            Average_Amount=("Amount", "mean")
        )
        .reset_index()
        .sort_values(
            "Total_Amount",
            ascending=False
        )
    )

else:

    source_system_summary = pd.DataFrame()

print("Total Amount   :", round(total_amount, 2))
print("Average Amount :", round(average_amount, 2))

print("\nStatus Summary:")
display(status_summary)

print("\nCity Summary:")
display(city_summary)

print("\nSource System Summary:")
display(source_system_summary)

# ------------------------------------------------------------
# 16. FINAL RELIABILITY STATUS
# ------------------------------------------------------------

if (
    overall_completeness >= 95
    and reconciliation_rate >= 95
    and len(mismatch_df) <= 5
):

    reliability_status = "RELIABLE"

elif (
    overall_completeness >= 90
    and reconciliation_rate >= 90
):

    reliability_status = "RELIABLE WITH MINOR ISSUES"

else:

    reliability_status = "REQUIRES REVIEW"

# ------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------

final_summary = pd.DataFrame({

    "Metric": [

        "Source Records",
        "Warehouse Records",
        "Matched Records",
        "Missing in Warehouse",
        "Warehouse Only",
        "Reconciliation Rate %",
        "Overall Completeness %",
        "Detail Mismatches",
        "Negative Amounts",
        "Missing Amounts",
        "Missing Event Dates",
        "Missing Last Updated",
        "Total Amount",
        "Average Amount",
        "Final Reliability Status"

    ],

    "Value": [

        source_count,
        warehouse_count,
        matched_count,
        len(missing_in_warehouse),
        len(warehouse_only),
        round(reconciliation_rate, 2),
        round(overall_completeness, 2),
        len(mismatch_df),
        negative_amounts,
        missing_amounts,
        missing_dates,
        missing_last_updated,
        round(total_amount, 2),
        round(average_amount, 2),
        reliability_status

    ]
})

print("\n" + "=" * 70)
print("FINAL RELIABILITY SIGN-OFF")
print("=" * 70)

display(final_summary)

# ------------------------------------------------------------
# 18. SAVE COMPLETE PYTHON OUTPUT
# ------------------------------------------------------------

output_file = "/content/Task_5_Python_Analysis.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    source_df.to_excel(
        writer,
        sheet_name="Cleaned_Source_Data",
        index=False
    )

    warehouse_df.to_excel(
        writer,
        sheet_name="Warehouse_Data",
        index=False
    )

    source_missing.to_excel(
        writer,
        sheet_name="Data_Quality_Report",
        index=False
    )

    reconciliation_detail.to_excel(
        writer,
        sheet_name="Reconciliation",
        index=False
    )

    comparison.to_excel(
        writer,
        sheet_name="Field_Comparison",
        index=False
    )

    mismatch_df.to_excel(
        writer,
        sheet_name="Mismatches",
        index=False
    )

    status_summary.to_excel(
        writer,
        sheet_name="Status_Summary",
        index=False
    )

    city_summary.to_excel(
        writer,
        sheet_name="City_Summary",
        index=False
    )

    source_system_summary.to_excel(
        writer,
        sheet_name="Source_System_Summary",
        index=False
    )

    final_summary.to_excel(
        writer,
        sheet_name="Reliability_Signoff",
        index=False
    )

print("\n" + "=" * 70)
print("✅ PYTHON ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 70)
print("Final output file:")
print(output_file)

print("\nNext step:")
print("Download Task_5_Python_Analysis.xlsx")

TASK 5 - PYTHON DATA ANALYSIS
✅ File found: /content/Cleaned_Dataset.xlsx

Available Sheets:
1. Cleaned_Source_Data
2. Raw_Warehouse_Data
3. Data_Dictionary

Source Sheet    : Cleaned_Source_Data
Warehouse Sheet : Raw_Warehouse_Data

Source Shape    : (300, 7)
Warehouse Shape : (299, 7)

Source Columns:
['Record_ID', 'Event_Date', 'city', 'Source_System', 'Amount', 'status', 'Last_Updated']

Warehouse Columns:
['Record_ID', 'Event_Date', 'City', 'Source_System', 'Amount', 'Status', 'Last_Updated']

✅ Data standardization completed.

1. DATA QUALITY CHECK
Source Records     : 300
Warehouse Records  : 299
Source Duplicates  : 0
Warehouse Duplicates: 0

Missing Value Report:


,Column,Missing_Values,Missing_%
0,Record_ID,0,0.00
1,Event_Date,0,0.00
2,city,0,0.00
3,Source_System,0,0.00
4,Amount,1,0.33
5,status,0,0.00
6,Last_Updated,1,0.33


Overall Data Completeness: 99.9 %

2. SOURCE VS WAREHOUSE RECONCILIATION
Source Records          : 299
Warehouse Records       : 299
Matched Records         : 296
Missing in Warehouse    : 3
Warehouse Only Records  : 3
Reconciliation Rate     : 99.0 %

3. DETAIL FIELD RECONCILIATION
Common fields being compared:
['Event_Date', 'Source_System', 'Amount']

Matched Detail Records: 292
Mismatched Detail Records: 5

4. DATA RISK CHECK
Negative Amounts       : 0
Missing Amounts        : 1
Missing Event Dates    : 0
Missing Last Updated   : 1

5. DATA FRESHNESS
Latest Last_Updated: 2026-07-31 04:00:00
Freshness Status: Available

6. BUSINESS SUMMARY
Total Amount   : 2268961.57
Average Amount : 7588.5

Status Summary:


""



City Summary:


...


Source System Summary:


,Source_System,Record_Count,Total_Amount,Average_Amount
2,Web,104,796107.76,7654.882308
0,Mobile App,102,770468.92,7628.405149
1,Partner API,94,702384.89,7472.179681



FINAL RELIABILITY SIGN-OFF


,Metric,Value
0,Source Records,299
1,Warehouse Records,299
2,Matched Records,296
3,Missing in Warehouse,3
4,Warehouse Only,3
5,Reconciliation Rate %,99.0
6,Overall Completeness %,99.9
7,Detail Mismatches,5
8,Negative Amounts,0
9,Missing Amounts,1



✅ PYTHON ANALYSIS COMPLETED SUCCESSFULLY
Final output file:
/content/Task_5_Python_Analysis.xlsx

Next step:
Download Task_5_Python_Analysis.xlsx
